# Script 2 - Chunking Strategies Comparison

#### Objective
This notebook transforms CV data into vector embeddings using  and evaluates 6 different chunking strategies stored in separate Qdrant in-memory collections:
1. **No Chunking**: Entire CV serialized into a single chunk using .
2. **Naive 3-Sections**: Functional section splitting constrained by  token limits.
3. **TokenChunker**: Chonkie TokenChunker (max 500 tokens, 100 overlap).
4. **WordChunker**: Chonkie WordChunker (max 500 tokens, 100 overlap).
5. **RecursiveChunker**: Chonkie RecursiveChunker (max 500 tokens).
6. **SemanticChunker**: Chonkie SemanticChunker.

Search quality across all 6 strategies is evaluated against the ground truth matrix using Hit Rate, MRR@2, MRR@3, and NDCG@10.

### 1. Initializing a local client

Initializing Qdrant in  mode for fast testing.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

### 2. Loading CV Data and Embedding Models

In [2]:
import os
import json
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

# Load CV data
with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

# Reference model for token limit constraints (Naive chunking)
ref_model_name = "avsolatorio/GIST-all-MiniLM-L6-v2"
ref_model = SentenceTransformer(ref_model_name, token=HUGGING_FACE_API_KEY, trust_remote_code=True)

# Embedding model
models_to_test = {
    "harrier-oss-v1-0.6b": {"size": 1024, "model_name": "microsoft/harrier-oss-v1-0.6b"},
}
model = SentenceTransformer(models_to_test["harrier-oss-v1-0.6b"]["model_name"], token=HUGGING_FACE_API_KEY)

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6527.44it/s]


### 3. Serialization Helper & Naive Chunking Function

In [3]:
def serialize_cv_for_embedding(cv: dict) -> str:
    """Formats CV text specifically for optimal embedding model vectorization."""
    certifications = ", ".join(cv.get("certifications", [])) or "None"
    skills = ", ".join(cv.get("skills", [])) or "None"
    languages = ", ".join(cv.get("languages", [])) or "None"
    
    experience = "\n".join([f"- {exp}" for exp in cv.get("experience", [])])
    education = "\n".join([f"- {edu}" for edu in cv.get("education", [])])

    text = f"""Candidate Name: {cv.get("name")}
Profession: {cv.get("profession")}
Seniority Level: {cv.get("seniority_level")}
Location: {cv.get("location")}
Years of Experience: {cv.get("experience_years")}
Languages: {languages}

About Me:
{cv.get("about_me", "")}

Skills:
{skills}

Certifications:
{certifications}

Work Experience:
{experience}

Education:
{education}
""".strip()
    
    return text


In [4]:
def is_less_than_max_tokens(ref_model, text, applicant_name, chunk_num):
    num_tokens = len(ref_model.tokenizer.encode(text))
    max_tokens = ref_model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

def get_naive_chunks(cv_data, ref_model):
    max_tokens = ref_model.max_seq_length
    data_points = []
    for idx, cv in enumerate(cv_data):
        # First CV chunk
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (about_me: {cv['about_me']}, seniority_level: {cv['seniority_level']}, location: {cv['location']}, experience_years: {cv['experience_years']}, languages: {cv.get('languages', [])})"
        )
        if is_less_than_max_tokens(ref_model, chunk, cv['name'], 1):
            data_points.append({"chunk": chunk, "chunk_meta": "about_me-seniority_level-location-experience_years-languages", "cv": cv})

        # Second CV chunk
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (certifications: {cv['certifications']}, education: {cv['education']}, skills: {cv['skills']})"
        )
        if is_less_than_max_tokens(ref_model, chunk, cv['name'], 2):
            data_points.append({"chunk": chunk, "chunk_meta": "certifications-education-skills", "cv": cv})
        else:
            chunk1 = (
                f"Candidate name: {cv['name']} | "
                f"Candidate profession: {cv['profession']} | "
                f"Content: (certifications: {cv['certifications']})"
            )
            data_points.append({"chunk": chunk1, "chunk_meta": "certifications", "cv": cv})
            
            chunk2 = (
                f"Candidate name: {cv['name']} | "
                f"Candidate profession: {cv['profession']} | "
                f"Content: (education: {cv['education']})"
            )
            data_points.append({"chunk": chunk2, "chunk_meta": "education", "cv": cv})
            
            chunk3 = (
                f"Candidate name: {cv['name']} | "
                f"Candidate profession: {cv['profession']} | "
                f"Content: (skills: {cv['skills']})"
            )
            data_points.append({"chunk": chunk3, "chunk_meta": "skills", "cv": cv})

        # Third chunk
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (experience: {cv['experience']})"
        )
        if is_less_than_max_tokens(ref_model, chunk, cv['name'], 3):
            data_points.append({"chunk": chunk, "chunk_meta": "experience", "cv": cv})
        else:
            exp = f"experience_1: {cv['experience']}"
            half = int(len(exp)/2)
            chunk1 = (
                f"Candidate name: {cv['name']} | "
                f"Candidate profession: {cv['profession']} | "
                f"Content: (experience_part_1: {exp[:half]})"
            )
            data_points.append({"chunk": chunk1, "chunk_meta": "experience_part_1", "cv": cv})
            
            chunk2 = (
                f"Candidate name: {cv['name']} | "
                f"Candidate profession: {cv['profession']} | "
                f"Content: (experience_part_2: {exp[half:]})"
            )
            data_points.append({"chunk": chunk2, "chunk_meta": "experience_part_2", "cv": cv})

    return data_points

### 4. Creating Collections, Vectorizing, and Uploading Chunks for 6 Strategies

In [5]:
from chonkie import TokenChunker, RecursiveChunker, SemanticChunker
try:
    from chonkie import WordChunker
except ImportError:
    from chonkie import SentenceChunker as WordChunker

# Initialize Chonkie chunkers
token_chunker = TokenChunker(chunk_size=500, chunk_overlap=100)
word_chunker = WordChunker(chunk_size=500, chunk_overlap=100)
recursive_chunker = RecursiveChunker(chunk_size=500)
semantic_chunker = SemanticChunker()

chunking_strategies = {
    "no_chunking": "no_chunking",
    "naive_3_sections": "naive",
    "token_chunker": token_chunker,
    "word_chunker": word_chunker,
    "recursive_chunker": recursive_chunker,
    "semantic_chunker": semantic_chunker,
}

strategy_collections = {}

for strategy_name, chunker in chunking_strategies.items():
    collection_name = f"collection_{strategy_name}"
    strategy_collections[strategy_name] = collection_name
    
    # 1. Create unique collection per strategy
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=models_to_test["harrier-oss-v1-0.6b"]["size"],
            distance=models.Distance.COSINE
        )
    )
    
    # 2. Chunk data
    data_points = []
    if strategy_name == "no_chunking":
        for cv in cv_data:
            cv_text = serialize_cv_for_embedding(cv)
            data_points.append({
                "chunk": cv_text,
                "cv": cv
            })
    elif strategy_name == "naive_3_sections":
        data_points = get_naive_chunks(cv_data, ref_model)
    else:
        for cv in cv_data:
            cv_text = serialize_cv_for_embedding(cv)
            chunks = chunker(cv_text)
            for c in chunks:
                chunk_str = c.text if hasattr(c, "text") else str(c)
                data_points.append({
                    "chunk": chunk_str,
                    "cv": cv
                })
                
    print(f"Strategy '{strategy_name}': generated {len(data_points)} total chunks.")
    
    # 3. Embed chunks
    embeddings = model.encode([dp["chunk"] for dp in data_points]).tolist()
    
    # 4. Upload points to Qdrant
    client.upload_points(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=idx,
                vector=embeddings[idx],
                payload={
                    **dp["cv"],
                    "chunk": dp["chunk"]
                }
            )
            for idx, dp in enumerate(data_points)
        ],
    )
    
    # 5. Index payload fields
    fields_to_index = ["location", "seniority_level", "email"]
    for field in fields_to_index:
        client.create_payload_index(
            collection_name=collection_name,
            field_name=field,
            field_schema=models.PayloadSchemaType.TEXT
        )

print("\nAll 6 collections created, embedded, and populated successfully!")


2026-08-04 20:52:12,741 | WARNING  | chonkie.embeddings.auto:get_embeddings:98 - Failed to load minishlab/potion-base-32M with Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to loading default provider model.
Traceback (most recent call last):
  File "/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/chonkie/embeddings/model2vec.py", line 32, in __init__
    from model2vec import StaticModel
ModuleNotFoundError: No module named 'model2vec'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/chonkie/embeddings/auto.py", line 96, in get_embeddings
    embeddings_instance = cast(Any, embeddings_cls)(model, **kwargs)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/col-ae-068/Documents/pe

Strategy 'no_chunking': generated 91 total chunks.


/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_62517/436175237.py:83: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (561 > 512). Running this sequence through the model will result in indexing errors


Strategy 'naive_3_sections': generated 303 total chunks.
Strategy 'token_chunker': generated 762 total chunks.
Strategy 'word_chunker': generated 790 total chunks.
Strategy 'recursive_chunker': generated 875 total chunks.
Strategy 'semantic_chunker': generated 864 total chunks.

All 6 collections created, embedded, and populated successfully!


### 5. Loading Job Descriptions & Ground Truth Evaluation Matrix

In [6]:
with open("../data/job_descriptions_train_batch.json", "r", encoding="utf-8") as f:
    job_descriptions = json.load(f)

with open("../data/ground_truth_matrix.json", "r", encoding="utf-8") as f:
    ground_truth_matrix = json.load(f)

### 6. Comparing Chunking Strategies Metrics

In [7]:
import math
import pandas as pd

Q = len(job_descriptions)
hr_thr = 2 # Hit rate relevance threshold
mrr2_thr = 2 # MRR_2 relevance threshold
mrr3_thr = 3 # MRR_3 relevance threshold
top_k = 10

metrics = {
    "hit_rate": {},
    "mrr_2": {}, # Mean Reciprocal Rank with a relevance score >= 2
    "mrr_3": {}, # Mean Reciprocal Rank with a relevance score >= 3
    "ndcg_10": {}, # Normalized Discounted Cumulative Gain at K=10
}

for strategy_name, collection_name in strategy_collections.items():

    hit_rate_accum = 0
    mrr2_accum = 0
    mrr3_accum = 0
    ndcg_accum = 0.0

    print("\n" + "="*80)
    print(f" Strategy Name: {strategy_name}")
    print("="*80)

    for job in job_descriptions:
        
        query_vector = model.encode(job["description"]).tolist()
        
        result = client.query_points_groups(
            collection_name=collection_name,
            query=query_vector,
            limit=top_k,
            group_by="email",
            group_size=1
        )

        has_hit = False
        mrr2_rr = 0.0
        mrr3_rr = 0.0
        dcg = 0.0

        for rank, group in enumerate(result.groups, start=1):
            eval_matches = [e for e in ground_truth_matrix[job["id"]]["evaluations"] if e["email"] == group.id]
            if eval_matches:
                benchmark_score = eval_matches[0]["score"]
                
                # Calculate DCG for current result
                dcg += benchmark_score / math.log2(rank + 1)
                
                if benchmark_score >= hr_thr:
                    has_hit = True
                
                if benchmark_score >= mrr2_thr and mrr2_rr == 0.0:
                    mrr2_rr = 1.0 / rank
                    
                if benchmark_score >= mrr3_thr and mrr3_rr == 0.0:
                    mrr3_rr = 1.0 / rank

        # Ideal DCG (IDCG) for current job from ground truth
        all_eval_scores = sorted(
            [e["score"] for e in ground_truth_matrix[job["id"]]["evaluations"]],
            reverse=True
        )[:top_k]
        idcg = sum(score / math.log2(rank + 1) for rank, score in enumerate(all_eval_scores, start=1))
        
        ndcg_query = (dcg / idcg) if idcg > 0 else 0.0
        ndcg_accum += ndcg_query

        if has_hit:
            hit_rate_accum += 1
            
        mrr2_accum += mrr2_rr
        mrr3_accum += mrr3_rr

    metrics["hit_rate"][strategy_name] = hit_rate_accum / Q
    metrics["mrr_2"][strategy_name] = mrr2_accum / Q
    metrics["mrr_3"][strategy_name] = mrr3_accum / Q
    metrics["ndcg_10"][strategy_name] = ndcg_accum / Q

df_metrics = pd.DataFrame(metrics)
df_metrics



 Strategy Name: no_chunking

 Strategy Name: naive_3_sections

 Strategy Name: token_chunker

 Strategy Name: word_chunker

 Strategy Name: recursive_chunker

 Strategy Name: semantic_chunker


,hit_rate,mrr_2,mrr_3,ndcg_10
no_chunking,1.0,0.840000,0.533333,0.782377
naive_3_sections,1.0,0.833333,0.500000,0.756539
token_chunker,1.0,0.595238,0.540000,0.744100
word_chunker,1.0,0.720000,0.566667,0.745326
recursive_chunker,1.0,0.666667,0.400000,0.727813
semantic_chunker,1.0,0.740000,0.406667,0.575573
